In [ ]:
# %% [markdown]
# # **Model Experimentation for Banking Fraud Detection**
# ## VeritasFinancial - Advanced Model Development
# 
# **Objective**: Develop and compare multiple modeling approaches for fraud detection, from classical ML to deep learning.
# 
# **Modeling Approaches**:
# 1. Classical ML (XGBoost, LightGBM, Random Forest)
# 2. Anomaly Detection (Isolation Forest, Autoencoders)
# 3. Deep Learning (Neural Networks, Transformers)
# 4. Ensemble Methods
# 5. Calibration and Threshold Optimization

# %% [markdown]
# ### **1. Environment Setup and Imports**

# %%
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
import os
import pickle
import json
warnings.filterwarnings('ignore')

# Machine Learning
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score,
    GridSearchCV, RandomizedSearchCV, TimeSeriesSplit
)
from sklearn.preprocessing import StandardScaler, RobustScaler, QuantileTransformer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, precision_recall_curve, roc_curve
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Classical ML Models
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    IsolationForest, AdaBoostClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import OneClassSVM

# Advanced ML Models
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

# Imbalanced learning
from imblearn.over_sampling import (
    SMOTE, ADASYN, RandomOverSampler, BorderlineSMOTE
)
from imblearn.under_sampling import (
    RandomUnderSampler, TomekLinks, EditedNearestNeighbours
)
from imblearn.combine import SMOTETomek, SMOTEENN
from imblearn.pipeline import Pipeline as ImbPipeline

# Deep Learning
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
import torch.nn.functional as F

# Hyperparameter optimization
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

# Explainability
import shap
import lime
import lime.lime_tabular
from eli5.sklearn import PermutationImportance

# Visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Set random seeds for reproducibility
def set_seed(seed=42):
    """Set all random seeds for reproducibility"""
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# %% [markdown]
# ### **2. Load Engineered Features**

# %%
class DataPreparator:
    """
    Prepare data for modeling with proper preprocessing and splitting.
    
    Handles:
    - Loading engineered features
    - Train/validation/test splits
    - Feature scaling
    - Class imbalance handling
    """
    
    def __init__(self, feature_path='../data/features/engineered_features.parquet'):
        """
        Initialize data preparator.
        
        Args:
            feature_path: Path to engineered features
        """
        self.feature_path = feature_path
        self.df = None
        self.X = None
        self.y = None
        self.feature_names = None
        self.scaler = None
        
    def load_data(self):
        """Load engineered features"""
        print("\n" + "="*60)
        print("LOADING ENGINEERED FEATURES")
        print("="*60)
        
        self.df = pd.read_parquet(self.feature_path)
        print(f"✅ Data loaded: {self.df.shape}")
        print(f"   Features: {self.df.shape[1] - 1} (excluding target)")
        print(f"   Samples: {self.df.shape[0]:,}")
        print(f"   Fraud rate: {self.df['is_fraud'].mean()*100:.4f}%")
        
        # Separate features and target
        self.y = self.df['is_fraud'].values
        self.X = self.df.drop(['is_fraud', 'transaction_id', 'customer_id'] 
                               if 'transaction_id' in self.df.columns 
                               else ['is_fraud'], axis=1)
        
        # Handle any remaining non-numeric columns
        self._handle_categorical_features()
        
        self.feature_names = self.X.columns.tolist()
        
        return self.X, self.y
    
    def _handle_categorical_features(self):
        """Handle categorical features"""
        categorical_cols = self.X.select_dtypes(include=['object', 'category']).columns
        
        if len(categorical_cols) > 0:
            print(f"\n📊 Found {len(categorical_cols)} categorical features:")
            for col in categorical_cols:
                print(f"   • {col}: {self.X[col].nunique()} unique values")
            
            # One-hot encode categorical features
            self.X = pd.get_dummies(self.X, columns=categorical_cols, drop_first=True)
            print(f"\n✅ After encoding: {self.X.shape[1]} features")
    
    def create_splits(self, test_size=0.2, val_size=0.25, stratify=True, time_based=False):
        """
        Create train/validation/test splits.
        
        Args:
            test_size: Proportion for test set
            val_size: Proportion of training for validation
            stratify: Whether to stratify by target
            time_based: Whether to use time-based split (for temporal validation)
            
        Returns:
            Dictionary with train/val/test splits
        """
        print("\n" + "="*60)
        print("CREATING DATA SPLITS")
        print("="*60)
        
        if time_based and 'transaction_time' in self.df.columns:
            # Time-based split (preserve temporal order)
            print("📅 Using time-based split...")
            sorted_idx = self.df['transaction_time'].sort_values().index
            self.X = self.X.loc[sorted_idx]
            self.y = self.y[sorted_idx]
            
            split_point = int(len(self.X) * (1 - test_size))
            val_point = int(split_point * (1 - val_size))
            
            X_train = self.X.iloc[:val_point]
            y_train = self.y[:val_point]
            X_val = self.X.iloc[val_point:split_point]
            y_val = self.y[val_point:split_point]
            X_test = self.X.iloc[split_point:]
            y_test = self.y[split_point:]
            
        else:
            # Random stratified split
            print("🎲 Using random stratified split...")
            # First split: training + validation vs test
            X_temp, X_test, y_temp, y_test = train_test_split(
                self.X, self.y,
                test_size=test_size,
                random_state=42,
                stratify=self.y if stratify else None
            )
            
            # Second split: training vs validation
            val_size_adjusted = val_size / (1 - test_size)
            X_train, X_val, y_train, y_val = train_test_split(
                X_temp, y_temp,
                test_size=val_size_adjusted,
                random_state=42,
                stratify=y_temp if stratify else None
            )
        
        self.splits = {
            'X_train': X_train, 'y_train': y_train,
            'X_val': X_val, 'y_val': y_val,
            'X_test': X_test, 'y_test': y_test
        }
        
        print(f"\n📊 Split Sizes:")
        print(f"   Training:   {len(X_train):,} ({len(y_train[y_train==1])} fraud, {len(y_train[y_train==0])} non-fraud)")
        print(f"   Validation: {len(X_val):,} ({len(y_val[y_val==1])} fraud, {len(y_val[y_val==0])} non-fraud)")
        print(f"   Test:       {len(X_test):,} ({len(y_test[y_test==1])} fraud, {len(y_test[y_test==0])} non-fraud)")
        
        return self.splits
    
    def scale_features(self, scaler_type='robust'):
        """
        Scale features using specified scaler.
        
        Args:
            scaler_type: Type of scaler ('standard', 'robust', 'quantile')
        """
        print("\n" + "="*60)
        print(f"SCALING FEATURES ({scaler_type})")
        print("="*60)
        
        if scaler_type == 'standard':
            self.scaler = StandardScaler()
        elif scaler_type == 'robust':
            self.scaler = RobustScaler()
        elif scaler_type == 'quantile':
            self.scaler = QuantileTransformer(output_distribution='normal')
        else:
            raise ValueError(f"Unknown scaler type: {scaler_type}")
        
        # Fit on training data only
        self.splits['X_train_scaled'] = self.scaler.fit_transform(self.splits['X_train'])
        self.splits['X_val_scaled'] = self.scaler.transform(self.splits['X_val'])
        self.splits['X_test_scaled'] = self.scaler.transform(self.splits['X_test'])
        
        print(f"✅ Features scaled successfully")
        print(f"   Training mean: {self.splits['X_train_scaled'].mean():.4f}")
        print(f"   Training std: {self.splits['X_train_scaled'].std():.4f}")
        
        return self.splits
    
    def handle_imbalance(self, method='smote', sampling_strategy='auto'):
        """
        Handle class imbalance using various resampling techniques.
        
        Args:
            method: Resampling method ('smote', 'adasyn', 'random_over', 'random_under', 'smote_tomek')
            sampling_strategy: Sampling strategy
        """
        print("\n" + "="*60)
        print(f"HANDLING CLASS IMBALANCE ({method})")
        print("="*60)
        
        print(f"Before resampling:")
        print(f"   Class 0: {np.sum(self.splits['y_train'] == 0)}")
        print(f"   Class 1: {np.sum(self.splits['y_train'] == 1)}")
        
        # Initialize resampler
        if method == 'smote':
            resampler = SMOTE(random_state=42, sampling_strategy=sampling_strategy)
        elif method == 'adasyn':
            resampler = ADASYN(random_state=42, sampling_strategy=sampling_strategy)
        elif method == 'borderline_smote':
            resampler = BorderlineSMOTE(random_state=42, sampling_strategy=sampling_strategy)
        elif method == 'random_over':
            resampler = RandomOverSampler(random_state=42, sampling_strategy=sampling_strategy)
        elif method == 'random_under':
            resampler = RandomUnderSampler(random_state=42, sampling_strategy=sampling_strategy)
        elif method == 'smote_tomek':
            resampler = SMOTETomek(random_state=42, sampling_strategy=sampling_strategy)
        elif method == 'smote_enn':
            resampler = SMOTEENN(random_state=42, sampling_strategy=sampling_strategy)
        else:
            print(f"⚠ Unknown method: {method}, skipping resampling")
            self.splits['X_train_resampled'] = self.splits['X_train_scaled']
            self.splits['y_train_resampled'] = self.splits['y_train']
            return self.splits
        
        # Apply resampling
        X_resampled, y_resampled = resampler.fit_resample(
            self.splits['X_train_scaled'], 
            self.splits['y_train']
        )
        
        self.splits['X_train_resampled'] = X_resampled
        self.splits['y_train_resampled'] = y_resampled
        
        print(f"\nAfter resampling:")
        print(f"   Class 0: {np.sum(y_resampled == 0)}")
        print(f"   Class 1: {np.sum(y_resampled == 1)}")
        
        return self.splits

# Initialize data preparator
preparator = DataPreparator()
X, y = preparator.load_data()
splits = preparator.create_splits(test_size=0.2, val_size=0.25, stratify=True)
splits = preparator.scale_features(scaler_type='robust')
splits = preparator.handle_imbalance(method='smote', sampling_strategy='auto')

# %% [markdown]
# ### **3. Classical ML Models**

# %%
class ClassicalMLExperimenter:
    """
    Experiment with classical machine learning models for fraud detection.
    
    Tests multiple algorithms with cross-validation and hyperparameter tuning.
    """
    
    def __init__(self, splits, feature_names):
        """
        Initialize classical ML experimenter.
        
        Args:
            splits: Dictionary with data splits
            feature_names: List of feature names
        """
        self.splits = splits
        self.feature_names = feature_names
        self.models = {}
        self.results = {}
        self.best_model = None
        self.best_score = 0
        
    def define_models(self):
        """Define baseline models to test"""
        print("\n" + "="*60)
        print("DEFINING BASELINE MODELS")
        print("="*60)
        
        self.models = {
            'Logistic Regression': LogisticRegression(
                class_weight='balanced',
                max_iter=1000,
                random_state=42
            ),
            'Random Forest': RandomForestClassifier(
                n_estimators=100,
                max_depth=10,
                class_weight='balanced',
                random_state=42,
                n_jobs=-1
            ),
            'XGBoost': xgb.XGBClassifier(
                n_estimators=100,
                max_depth=6,
                learning_rate=0.1,
                scale_pos_weight=(len(self.splits['y_train'][self.splits['y_train']==0]) / 
                                  len(self.splits['y_train'][self.splits['y_train']==1])),
                random_state=42,
                use_label_encoder=False,
                eval_metric='logloss'
            ),
            'LightGBM': lgb.LGBMClassifier(
                n_estimators=100,
                max_depth=6,
                learning_rate=0.1,
                class_weight='balanced',
                random_state=42,
                verbose=-1
            ),
            'CatBoost': cb.CatBoostClassifier(
                iterations=100,
                depth=6,
                learning_rate=0.1,
                auto_class_weights='Balanced',
                random_seed=42,
                verbose=0
            ),
            'Gradient Boosting': GradientBoostingClassifier(
                n_estimators=100,
                max_depth=6,
                learning_rate=0.1,
                random_state=42
            ),
            'AdaBoost': AdaBoostClassifier(
                n_estimators=100,
                learning_rate=0.1,
                random_state=42
            )
        }
        
        for name, model in self.models.items():
            print(f"   ✓ {name}")
        
        return self.models
    
    def train_and_evaluate_all(self, cv_folds=5):
        """
        Train and evaluate all baseline models.
        
        Args:
            cv_folds: Number of cross-validation folds
        """
        print("\n" + "="*60)
        print("TRAINING AND EVALUATING ALL MODELS")
        print("="*60)
        
        X_train = self.splits['X_train_resampled']
        y_train = self.splits['y_train_resampled']
        X_val = self.splits['X_val_scaled']
        y_val = self.splits['y_val']
        
        for name, model in self.models.items():
            print(f"\n📊 Training {name}...")
            
            try:
                # Train model
                start_time = datetime.now()
                model.fit(X_train, y_train)
                training_time = (datetime.now() - start_time).total_seconds()
                
                # Predictions
                y_pred = model.predict(X_val)
                y_pred_proba = model.predict_proba(X_val)[:, 1]
                
                # Calculate metrics
                metrics = self._calculate_metrics(y_val, y_pred, y_pred_proba)
                metrics['training_time'] = training_time
                
                # Cross-validation score
                if cv_folds > 1:
                    cv_scores = cross_val_score(
                        model, X_train, y_train, 
                        cv=StratifiedKFold(cv_folds), 
                        scoring='roc_auc'
                    )
                    metrics['cv_mean'] = cv_scores.mean()
                    metrics['cv_std'] = cv_scores.std()
                
                self.results[name] = metrics
                
                print(f"   ✓ F1: {metrics['f1']:.4f}, AUC-ROC: {metrics['roc_auc']:.4f}, "
                      f"Time: {training_time:.2f}s")
                
                # Track best model
                if metrics['f1'] > self.best_score:
                    self.best_score = metrics['f1']
                    self.best_model = model
                    self.best_model_name = name
                    
            except Exception as e:
                print(f"   ✗ Error: {str(e)}")
                self.results[name] = {'error': str(e)}
        
        return self.results
    
    def _calculate_metrics(self, y_true, y_pred, y_pred_proba):
        """Calculate comprehensive metrics"""
        return {
            'accuracy': accuracy_score(y_true, y_pred),
            'precision': precision_score(y_true, y_pred, zero_division=0),
            'recall': recall_score(y_true, y_pred, zero_division=0),
            'f1': f1_score(y_true, y_pred, zero_division=0),
            'roc_auc': roc_auc_score(y_true, y_pred_proba),
            'pr_auc': average_precision_score(y_true, y_pred_proba)
        }
    
    def compare_results(self):
        """Compare results from all models"""
        print("\n" + "="*60)
        print("MODEL COMPARISON RESULTS")
        print("="*60)
        
        # Create comparison DataFrame
        comparison = pd.DataFrame(self.results).T
        comparison = comparison.sort_values('f1', ascending=False)
        
        print("\n📊 Performance Summary (sorted by F1 Score):")
        print(comparison[['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc']].round(4))
        
        # Identify best model
        print(f"\n🏆 Best Model: {self.best_model_name}")
        print(f"   F1 Score: {self.best_score:.4f}")
        
        return comparison
    
    def plot_comparison(self):
        """Visualize model comparison"""
        comparison = pd.DataFrame(self.results).T
        
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        
        # F1 Score
        axes[0, 0].barh(comparison.index, comparison['f1'])
        axes[0, 0].set_xlabel('F1 Score')
        axes[0, 0].set_title('F1 Score by Model')
        axes[0, 0].axvline(x=comparison['f1'].max(), color='red', linestyle='--', alpha=0.5)
        
        # ROC AUC
        axes[0, 1].barh(comparison.index, comparison['roc_auc'])
        axes[0, 1].set_xlabel('ROC AUC')
        axes[0, 1].set_title('ROC AUC by Model')
        axes[0, 1].axvline(x=comparison['roc_auc'].max(), color='red', linestyle='--', alpha=0.5)
        
        # Precision-Recall
        axes[1, 0].barh(comparison.index, comparison['pr_auc'])
        axes[1, 0].set_xlabel('PR AUC')
        axes[1, 0].set_title('Precision-Recall AUC by Model')
        axes[1, 0].axvline(x=comparison['pr_auc'].max(), color='red', linestyle='--', alpha=0.5)
        
        # Training Time
        axes[1, 1].barh(comparison.index, comparison['training_time'])
        axes[1, 1].set_xlabel('Training Time (seconds)')
        axes[1, 1].set_title('Training Time by Model')
        axes[1, 1].set_xscale('log')
        
        plt.tight_layout()
        plt.show()

# Run classical ML experiments
classical_exp = ClassicalMLExperimenter(splits, preparator.feature_names)
classical_exp.define_models()
results = classical_exp.train_and_evaluate_all(cv_folds=5)
comparison = classical_exp.compare_results()
classical_exp.plot_comparison()

# %% [markdown]
# ### **4. Advanced XGBoost with Hyperparameter Optimization**

# %%
class XGBoostOptimizer:
    """
    Advanced XGBoost with hyperparameter optimization using Optuna.
    
    Performs:
    - Bayesian hyperparameter optimization
    - Cross-validation
    - Feature importance analysis
    - Threshold optimization
    """
    
    def __init__(self, splits):
        """
        Initialize XGBoost optimizer.
        
        Args:
            splits: Dictionary with data splits
        """
        self.splits = splits
        self.best_params = None
        self.best_model = None
        self.study = None
        self.feature_importance = None
        
    def objective(self, trial):
        """
        Optuna objective function for XGBoost optimization.
        
        Args:
            trial: Optuna trial object
            
        Returns:
            Validation F1 score
        """
        # Define hyperparameter search space
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=100),
            'max_depth': trial.suggest_int('max_depth', 3, 12),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.6, 1.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'gamma': trial.suggest_float('gamma', 0, 5),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
            'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1, 50),
        }
        
        # Calculate class weight based on imbalance
        neg_count = len(self.splits['y_train'][self.splits['y_train'] == 0])
        pos_count = len(self.splits['y_train'][self.splits['y_train'] == 1])
        default_scale = neg_count / pos_count
        
        # Use suggested scale_pos_weight or default
        params['scale_pos_weight'] = trial.suggest_float('scale_pos_weight', default_scale * 0.5, default_scale * 2)
        
        # Train model
        model = xgb.XGBClassifier(
            **params,
            random_state=42,
            use_label_encoder=False,
            eval_metric='logloss',
            early_stopping_rounds=50,
            n_jobs=-1
        )
        
        # Cross-validation
        X_train = self.splits['X_train_resampled']
        y_train = self.splits['y_train_resampled']
        X_val = self.splits['X_val_scaled']
        y_val = self.splits['y_val']
        
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )
        
        # Predict and evaluate
        y_pred = model.predict(X_val)
        f1 = f1_score(y_val, y_pred)
        
        return f1
    
    def optimize(self, n_trials=50):
        """
        Run hyperparameter optimization.
        
        Args:
            n_trials: Number of optimization trials
        """
        print("\n" + "="*60)
        print("XGBOOST HYPERPARAMETER OPTIMIZATION")
        print("="*60)
        
        # Create Optuna study
        self.study = optuna.create_study(
            direction='maximize',
            sampler=TPESampler(seed=42),
            pruner=MedianPruner(n_startup_trials=10, n_warmup_steps=20)
        )
        
        # Run optimization
        self.study.optimize(self.objective, n_trials=n_trials, show_progress_bar=True)
        
        # Get best parameters
        self.best_params = self.study.best_params
        
        print(f"\n🏆 Best Trial: {self.study.best_trial.number}")
        print(f"   Best F1 Score: {self.study.best_value:.4f}")
        print(f"\nBest Parameters:")
        for param, value in self.best_params.items():
            print(f"   {param}: {value}")
        
        return self.best_params
    
    def train_best_model(self):
        """Train model with best parameters"""
        print("\n" + "="*60)
        print("TRAINING BEST XGBOOST MODEL")
        print("="*60)
        
        # Prepare data
        X_train = self.splits['X_train_resampled']
        y_train = self.splits['y_train_resampled']
        X_val = self.splits['X_val_scaled']
        y_val = self.splits['y_val']
        
        # Train model
        self.best_model = xgb.XGBClassifier(
            **self.best_params,
            random_state=42,
            use_label_encoder=False,
            eval_metric='logloss',
            early_stopping_rounds=50,
            n_jobs=-1
        )
        
        self.best_model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=True
        )
        
        # Feature importance
        self.feature_importance = pd.DataFrame({
            'feature': preparator.feature_names,
            'importance': self.best_model.feature_importances_
        }).sort_values('importance', ascending=False)
        
        print("\n📊 Top 10 Most Important Features:")
        for i, row in self.feature_importance.head(10).iterrows():
            print(f"   {i+1}. {row['feature']}: {row['importance']:.4f}")
        
        return self.best_model
    
    def plot_optimization_history(self):
        """Plot optimization history"""
        if self.study is None:
            print("No optimization history available")
            return
        
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))
        
        # Optimization history
        trials_df = self.study.trials_dataframe()
        axes[0].plot(trials_df['number'], trials_df['value'], 'o-', alpha=0.7)
        axes[0].axhline(y=self.study.best_value, color='red', linestyle='--', 
                        label=f"Best: {self.study.best_value:.4f}")
        axes[0].set_xlabel('Trial Number')
        axes[0].set_ylabel('F1 Score')
        axes[0].set_title('Optimization History')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        
        # Parameter importance
        if len(self.study.trials) > 10:
            param_importance = optuna.importance.get_param_importances(self.study)
            params = list(param_importance.keys())
            importance = list(param_importance.values())
            
            axes[1].barh(params, importance)
            axes[1].set_xlabel('Importance')
            axes[1].set_title('Hyperparameter Importance')
        
        plt.tight_layout()
        plt.show()
    
    def plot_feature_importance(self, top_n=20):
        """Plot feature importance"""
        if self.feature_importance is None:
            print("No feature importance available")
            return
        
        plt.figure(figsize=(10, 8))
        top_features = self.feature_importance.head(top_n)
        plt.barh(range(len(top_features)), top_features['importance'])
        plt.yticks(range(len(top_features)), top_features['feature'])
        plt.xlabel('Importance')
        plt.title(f'Top {top_n} Feature Importance')
        plt.gca().invert_yaxis()
        plt.tight_layout()
        plt.show()

# Run XGBoost optimization
xgb_optimizer = XGBoostOptimizer(splits)
best_params = xgb_optimizer.optimize(n_trials=30)
best_model = xgb_optimizer.train_best_model()
xgb_optimizer.plot_optimization_history()
xgb_optimizer.plot_feature_importance()

# %% [markdown]
# ### **5. Anomaly Detection Models**

# %%
class AnomalyDetectionExperimenter:
    """
    Experiment with unsupervised anomaly detection models.
    
    Tests:
    - Isolation Forest
    - One-Class SVM
    - Local Outlier Factor
    - Elliptic Envelope
    - Autoencoder (Deep Learning)
    """
    
    def __init__(self, splits):
        """
        Initialize anomaly detection experimenter.
        
        Args:
            splits: Dictionary with data splits
        """
        self.splits = splits
        self.models = {}
        self.results = {}
        
    def define_models(self):
        """Define anomaly detection models"""
        print("\n" + "="*60)
        print("DEFINING ANOMALY DETECTION MODELS")
        print("="*60)
        
        # Calculate contamination (expected fraud rate)
        contamination = self.splits['y_train'].mean()
        print(f"Expected contamination: {contamination:.4f}")
        
        self.models = {
            'Isolation Forest': IsolationForest(
                contamination=contamination,
                random_state=42,
                n_estimators=100,
                max_samples='auto',
                bootstrap=False,
                n_jobs=-1
            ),
            'One-Class SVM': OneClassSVM(
                nu=contamination,
                kernel='rbf',
                gamma='scale'
            ),
            'Local Outlier Factor': LocalOutlierFactor(
                contamination=contamination,
                n_neighbors=20,
                algorithm='auto',
                leaf_size=30,
                metric='minkowski',
                p=2,
                novelty=True  # Enable predict on new data
            ),
            'Elliptic Envelope': EllipticEnvelope(
                contamination=contamination,
                random_state=42,
                support_fraction=0.7
            )
        }
        
        for name in self.models.keys():
            print(f"   ✓ {name}")
        
        return self.models
    
    def train_and_evaluate(self):
        """Train and evaluate anomaly detection models"""
        print("\n" + "="*60)
        print("TRAINING ANOMALY DETECTION MODELS")
        print("="*60)
        
        X_train = self.splits['X_train_scaled']  # Use original training (not resampled)
        X_val = self.splits['X_val_scaled']
        y_val = self.splits['y_val']
        
        for name, model in self.models.items():
            print(f"\n📊 Training {name}...")
            
            try:
                # Train model
                start_time = datetime.now()
                
                if name == 'Local Outlier Factor':
                    # LOF requires special handling
                    model.fit(X_train)
                    # For novelty detection
                    model = LocalOutlierFactor(
                        contamination=model.contamination,
                        n_neighbors=model.n_neighbors,
                        novelty=True
                    )
                    model.fit(X_train)
                else:
                    model.fit(X_train)
                
                training_time = (datetime.now() - start_time).total_seconds()
                
                # Predictions (anomaly detection returns -1 for outliers, 1 for inliers)
                y_pred_raw = model.predict(X_val)
                # Convert to fraud detection format (1 for fraud/anomaly, 0 for normal)
                y_pred = (y_pred_raw == -1).astype(int)
                
                # Get anomaly scores (if available)
                if hasattr(model, 'score_samples'):
                    anomaly_scores = -model.score_samples(X_val)  # Higher = more anomalous
                elif hasattr(model, 'decision_function'):
                    anomaly_scores = -model.decision_function(X_val)
                else:
                    anomaly_scores = y_pred_raw  # Fallback
                
                # Calculate metrics
                metrics = {
                    'accuracy': accuracy_score(y_val, y_pred),
                    'precision': precision_score(y_val, y_pred, zero_division=0),
                    'recall': recall_score(y_val, y_pred, zero_division=0),
                    'f1': f1_score(y_val, y_pred, zero_division=0),
                    'training_time': training_time
                }
                
                # Calculate ROC AUC if possible
                try:
                    metrics['roc_auc'] = roc_auc_score(y_val, anomaly_scores)
                    metrics['pr_auc'] = average_precision_score(y_val, anomaly_scores)
                except:
                    metrics['roc_auc'] = 0
                    metrics['pr_auc'] = 0
                
                self.results[name] = metrics
                
                print(f"   ✓ F1: {metrics['f1']:.4f}, Recall: {metrics['recall']:.4f}, "
                      f"Time: {training_time:.2f}s")
                
            except Exception as e:
                print(f"   ✗ Error: {str(e)}")
                self.results[name] = {'error': str(e)}
        
        return self.results
    
    def compare_results(self):
        """Compare anomaly detection results"""
        print("\n" + "="*60)
        print("ANOMALY DETECTION RESULTS")
        print("="*60)
        
        comparison = pd.DataFrame(self.results).T
        comparison = comparison.sort_values('f1', ascending=False)
        
        print("\n📊 Performance Summary:")
        print(comparison[['precision', 'recall', 'f1', 'roc_auc']].round(4))
        
        return comparison

# Run anomaly detection experiments
anomaly_exp = AnomalyDetectionExperimenter(splits)
anomaly_exp.define_models()
anomaly_results = anomaly_exp.train_and_evaluate()
anomaly_comparison = anomaly_exp.compare_results()

# %% [markdown]
# ### **6. Deep Learning Models**

# %%
class DeepLearningExperimenter:
    """
    Experiment with deep learning models for fraud detection.
    
    Implements:
    - Feedforward Neural Network
    - Autoencoder for anomaly detection
    - Transformer model
    - LSTM for sequence modeling
    """
    
    def __init__(self, splits, input_dim):
        """
        Initialize deep learning experimenter.
        
        Args:
            splits: Dictionary with data splits
            input_dim: Input dimension
        """
        self.splits = splits
        self.input_dim = input_dim
        self.device = device
        self.models = {}
        self.results = {}
        
    def create_data_loaders(self, batch_size=256):
        """
        Create PyTorch data loaders.
        
        Args:
            batch_size: Batch size for training
        """
        # Convert to tensors
        X_train = torch.FloatTensor(self.splits['X_train_resampled']).to(self.device)
        y_train = torch.FloatTensor(self.splits['y_train_resampled']).to(self.device)
        X_val = torch.FloatTensor(self.splits['X_val_scaled']).to(self.device)
        y_val = torch.FloatTensor(self.splits['y_val']).to(self.device)
        X_test = torch.FloatTensor(self.splits['X_test_scaled']).to(self.device)
        y_test = torch.FloatTensor(self.splits['y_test']).to(self.device)
        
        # Create datasets
        train_dataset = TensorDataset(X_train, y_train)
        val_dataset = TensorDataset(X_val, y_val)
        test_dataset = TensorDataset(X_test, y_test)
        
        # Handle class imbalance with weighted sampler
        class_counts = np.bincount(self.splits['y_train_resampled'].astype(int))
        class_weights = 1. / class_counts
        sample_weights = class_weights[self.splits['y_train_resampled'].astype(int)]
        sampler = WeightedRandomSampler(
            weights=sample_weights,
            num_samples=len(sample_weights),
            replacement=True
        )
        
        # Create data loaders
        self.train_loader = DataLoader(
            train_dataset, 
            batch_size=batch_size, 
            sampler=sampler,
            pin_memory=True if self.device == 'cuda' else False
        )
        self.val_loader = DataLoader(
            val_dataset, 
            batch_size=batch_size, 
            shuffle=False,
            pin_memory=True if self.device == 'cuda' else False
        )
        self.test_loader = DataLoader(
            test_dataset, 
            batch_size=batch_size, 
            shuffle=False,
            pin_memory=True if self.device == 'cuda' else False
        )
        
        print(f"\n📊 Data Loaders Created:")
        print(f"   Training batches: {len(self.train_loader)}")
        print(f"   Validation batches: {len(self.val_loader)}")
        print(f"   Test batches: {len(self.test_loader)}")
        
    class FeedforwardNN(nn.Module):
        """Feedforward Neural Network for fraud detection"""
        
        def __init__(self, input_dim, hidden_dims=[256, 128, 64], dropout_rate=0.3):
            super().__init__()
            
            layers = []
            prev_dim = input_dim
            
            for hidden_dim in hidden_dims:
                layers.extend([
                    nn.Linear(prev_dim, hidden_dim),
                    nn.BatchNorm1d(hidden_dim),
                    nn.ReLU(),
                    nn.Dropout(dropout_rate)
                ])
                prev_dim = hidden_dim
            
            layers.append(nn.Linear(prev_dim, 1))
            layers.append(nn.Sigmoid())
            
            self.network = nn.Sequential(*layers)
            
        def forward(self, x):
            return self.network(x)
    
    class Autoencoder(nn.Module):
        """Autoencoder for anomaly detection"""
        
        def __init__(self, input_dim, encoding_dim=32):
            super().__init__()
            
            # Encoder
            self.encoder = nn.Sequential(
                nn.Linear(input_dim, 128),
                nn.ReLU(),
                nn.Linear(128, 64),
                nn.ReLU(),
                nn.Linear(64, encoding_dim),
                nn.ReLU()
            )
            
            # Decoder
            self.decoder = nn.Sequential(
                nn.Linear(encoding_dim, 64),
                nn.ReLU(),
                nn.Linear(64, 128),
                nn.ReLU(),
                nn.Linear(128, input_dim)
            )
            
        def forward(self, x):
            encoded = self.encoder(x)
            decoded = self.decoder(encoded)
            return decoded, encoded
        
        def get_anomaly_score(self, x):
            """Calculate reconstruction error as anomaly score"""
            reconstructed, _ = self.forward(x)
            mse = F.mse_loss(reconstructed, x, reduction='none').mean(dim=1)
            return mse
    
    class FraudTransformer(nn.Module):
        """Transformer model for fraud detection"""
        
        def __init__(self, input_dim, d_model=128, nhead=8, num_layers=4, dropout=0.1):
            super().__init__()
            
            self.input_projection = nn.Linear(input_dim, d_model)
            
            encoder_layer = nn.TransformerEncoderLayer(
                d_model=d_model,
                nhead=nhead,
                dim_feedforward=512,
                dropout=dropout,
                activation='relu',
                batch_first=True
            )
            self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
            
            self.output_layer = nn.Sequential(
                nn.Linear(d_model, 64),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(64, 1),
                nn.Sigmoid()
            )
            
        def forward(self, x):
            # Add sequence dimension (treat as sequence of 1)
            x = x.unsqueeze(1)  # [batch, 1, features]
            x = self.input_projection(x)
            x = self.transformer(x)
            x = x.mean(dim=1)  # Global average pooling
            return self.output_layer(x)
    
    def train_model(self, model, model_name, criterion, optimizer, scheduler=None, epochs=50):
        """Train a PyTorch model"""
        
        print(f"\n📊 Training {model_name}...")
        
        train_losses = []
        val_losses = []
        val_f1_scores = []
        best_val_f1 = 0
        patience_counter = 0
        patience = 10
        
        for epoch in range(epochs):
            # Training
            model.train()
            train_loss = 0
            for batch_X, batch_y in self.train_loader:
                optimizer.zero_grad()
                
                if model_name == 'Autoencoder':
                    # Autoencoder uses input as target
                    reconstructed, _ = model(batch_X)
                    loss = criterion(reconstructed, batch_X)
                else:
                    outputs = model(batch_X).squeeze()
                    loss = criterion(outputs, batch_y)
                
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                
                if scheduler:
                    scheduler.step()
                
                train_loss += loss.item()
            
            avg_train_loss = train_loss / len(self.train_loader)
            train_losses.append(avg_train_loss)
            
            # Validation
            model.eval()
            val_loss = 0
            all_preds = []
            all_labels = []
            
            with torch.no_grad():
                for batch_X, batch_y in self.val_loader:
                    if model_name == 'Autoencoder':
                        reconstructed, _ = model(batch_X)
                        loss = criterion(reconstructed, batch_X)
                        
                        # For autoencoder, use reconstruction error as anomaly score
                        recon_error = F.mse_loss(reconstructed, batch_X, reduction='none').mean(dim=1)
                        preds = (recon_error > recon_error.median()).float()
                    else:
                        outputs = model(batch_X).squeeze()
                        loss = criterion(outputs, batch_y)
                        preds = (outputs > 0.5).float()
                    
                    val_loss += loss.item()
                    all_preds.extend(preds.cpu().numpy())
                    all_labels.extend(batch_y.cpu().numpy())
            
            avg_val_loss = val_loss / len(self.val_loader)
            val_losses.append(avg_val_loss)
            
            # Calculate F1 score
            val_f1 = f1_score(all_labels, all_preds, zero_division=0)
            val_f1_scores.append(val_f1)
            
            # Early stopping
            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                torch.save(model.state_dict(), f'../artifacts/models/best_{model_name.lower()}.pt')
                patience_counter = 0
            else:
                patience_counter += 1
            
            if patience_counter >= patience:
                print(f"   Early stopping at epoch {epoch+1}")
                break
            
            if (epoch + 1) % 10 == 0:
                print(f"   Epoch {epoch+1}/{epochs}: Train Loss: {avg_train_loss:.4f}, "
                      f"Val Loss: {avg_val_loss:.4f}, Val F1: {val_f1:.4f}")
        
        # Load best model
        model.load_state_dict(torch.load(f'../artifacts/models/best_{model_name.lower()}.pt'))
        
        return model, {'train_losses': train_losses, 'val_losses': val_losses, 
                       'val_f1_scores': val_f1_scores, 'best_val_f1': best_val_f1}
    
    def run_experiments(self):
        """Run all deep learning experiments"""
        
        # Create data loaders
        self.create_data_loaders()
        
        # Calculate class weights for imbalanced data
        pos_weight = len(self.splits['y_train_resampled'][self.splits['y_train_resampled']==0]) / \
                     len(self.splits['y_train_resampled'][self.splits['y_train_resampled']==1])
        
        # Experiment 1: Feedforward Neural Network
        print("\n" + "="*60)
        print("EXPERIMENT 1: FEEDFORWARD NEURAL NETWORK")
        print("="*60)
        
        ff_model = self.FeedforwardNN(self.input_dim).to(self.device)
        ff_criterion = nn.BCELoss()
        ff_optimizer = optim.Adam(ff_model.parameters(), lr=0.001, weight_decay=1e-5)
        ff_scheduler = optim.lr_scheduler.OneCycleLR(
            ff_optimizer, max_lr=0.01, epochs=50, 
            steps_per_epoch=len(self.train_loader)
        )
        
        ff_model, ff_history = self.train_model(
            ff_model, 'FeedforwardNN', ff_criterion, ff_optimizer, ff_scheduler, epochs=50
        )
        self.models['FeedforwardNN'] = ff_model
        self.results['FeedforwardNN'] = ff_history
        
        # Experiment 2: Autoencoder
        print("\n" + "="*60)
        print("EXPERIMENT 2: AUTOENCODER")
        print("="*60)
        
        ae_model = self.Autoencoder(self.input_dim, encoding_dim=32).to(self.device)
        ae_criterion = nn.MSELoss()
        ae_optimizer = optim.Adam(ae_model.parameters(), lr=0.001)
        
        ae_model, ae_history = self.train_model(
            ae_model, 'Autoencoder', ae_criterion, ae_optimizer, epochs=50
        )
        self.models['Autoencoder'] = ae_model
        self.results['Autoencoder'] = ae_history
        
        # Experiment 3: Transformer
        print("\n" + "="*60)
        print("EXPERIMENT 3: TRANSFORMER")
        print("="*60)
        
        transformer_model = self.FraudTransformer(self.input_dim).to(self.device)
        transformer_criterion = nn.BCELoss()
        transformer_optimizer = optim.AdamW(transformer_model.parameters(), lr=0.0001, weight_decay=0.01)
        
        transformer_model, transformer_history = self.train_model(
            transformer_model, 'Transformer', transformer_criterion, transformer_optimizer, epochs=50
        )
        self.models['Transformer'] = transformer_model
        self.results['Transformer'] = transformer_history
        
        return self.models, self.results
    
    def plot_training_history(self):
        """Plot training history for all models"""
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        
        for idx, (name, history) in enumerate(self.results.items()):
            row, col = idx // 2, idx % 2
            
            if row < 2 and col < 2:
                axes[row, col].plot(history['train_losses'], label='Train Loss')
                axes[row, col].plot(history['val_losses'], label='Val Loss')
                axes[row, col].set_title(f'{name} - Loss')
                axes[row, col].set_xlabel('Epoch')
                axes[row, col].set_ylabel('Loss')
                axes[row, col].legend()
                axes[row, col].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Plot F1 scores
        plt.figure(figsize=(10, 6))
        for name, history in self.results.items():
            plt.plot(history['val_f1_scores'], label=name)
        plt.xlabel('Epoch')
        plt.ylabel('Validation F1 Score')
        plt.title('Model Comparison - Validation F1 Score')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()

# Run deep learning experiments
dl_exp = DeepLearningExperimenter(splits, X_train.shape[1])
dl_models, dl_results = dl_exp.run_experiments()
dl_exp.plot_training_history()

# %% [markdown]
# ### **7. Ensemble Methods**

# %%
class EnsembleExperimenter:
    """
    Create and evaluate ensemble models.
    
    Implements:
    - Voting Ensemble (hard/soft voting)
    - Stacking Ensemble
    - Blending Ensemble
    """
    
    def __init__(self, splits, base_models):
        """
        Initialize ensemble experimenter.
        
        Args:
            splits: Dictionary with data splits
            base_models: Dictionary of base models
        """
        self.splits = splits
        self.base_models = base_models
        self.ensemble_models = {}
        self.results = {}
        
    def create_voting_ensemble(self):
        """Create voting ensemble"""
        from sklearn.ensemble import VotingClassifier
        
        print("\n" + "="*60)
        print("CREATING VOTING ENSEMBLE")
        print("="*60)
        
        # Prepare estimators list
        estimators = [(name, model) for name, model in self.base_models.items()]
        
        # Hard voting (majority vote)
        hard_voting = VotingClassifier(
            estimators=estimators,
            voting='hard',
            n_jobs=-1
        )
        
        # Soft voting (weighted probabilities)
        soft_voting = VotingClassifier(
            estimators=estimators,
            voting='soft',
            weights=[self.results.get(name, {}).get('f1', 1) for name in self.base_models.keys()],
            n_jobs=-1
        )
        
        self.ensemble_models['Hard Voting'] = hard_voting
        self.ensemble_models['Soft Voting'] = soft_voting
        
        print(f"   ✓ Hard Voting Ensemble created with {len(estimators)} models")
        print(f"   ✓ Soft Voting Ensemble created with {len(estimators)} models")
        
    def create_stacking_ensemble(self):
        """Create stacking ensemble"""
        from sklearn.ensemble import StackingClassifier
        from sklearn.linear_model import LogisticRegression
        
        print("\n" + "="*60)
        print("CREATING STACKING ENSEMBLE")
        print("="*60)
        
        # Prepare estimators list
        estimators = [(name, model) for name, model in self.base_models.items()]
        
        # Meta-learner
        meta_learner = LogisticRegression(
            class_weight='balanced',
            max_iter=1000,
            random_state=42
        )
        
        # Stacking ensemble
        stacking = StackingClassifier(
            estimators=estimators,
            final_estimator=meta_learner,
            cv=5,
            stack_method='predict_proba',
            n_jobs=-1,
            passthrough=False
        )
        
        self.ensemble_models['Stacking'] = stacking
        print(f"   ✓ Stacking Ensemble created with {len(estimators)} base models")
        
    def train_and_evaluate(self):
        """Train and evaluate ensemble models"""
        print("\n" + "="*60)
        print("TRAINING ENSEMBLE MODELS")
        print("="*60)
        
        X_train = self.splits['X_train_resampled']
        y_train = self.splits['y_train_resampled']
        X_val = self.splits['X_val_scaled']
        y_val = self.splits['y_val']
        
        for name, ensemble in self.ensemble_models.items():
            print(f"\n📊 Training {name}...")
            
            try:
                start_time = datetime.now()
                ensemble.fit(X_train, y_train)
                training_time = (datetime.now() - start_time).total_seconds()
                
                # Predictions
                y_pred = ensemble.predict(X_val)
                y_pred_proba = ensemble.predict_proba(X_val)[:, 1]
                
                # Metrics
                metrics = {
                    'accuracy': accuracy_score(y_val, y_pred),
                    'precision': precision_score(y_val, y_pred, zero_division=0),
                    'recall': recall_score(y_val, y_pred, zero_division=0),
                    'f1': f1_score(y_val, y_pred, zero_division=0),
                    'roc_auc': roc_auc_score(y_val, y_pred_proba),
                    'pr_auc': average_precision_score(y_val, y_pred_proba),
                    'training_time': training_time
                }
                
                self.results[name] = metrics
                
                print(f"   ✓ F1: {metrics['f1']:.4f}, AUC-ROC: {metrics['roc_auc']:.4f}")
                
            except Exception as e:
                print(f"   ✗ Error: {str(e)}")
                self.results[name] = {'error': str(e)}
        
        return self.results
    
    def compare_results(self):
        """Compare ensemble results"""
        print("\n" + "="*60)
        print("ENSEMBLE RESULTS COMPARISON")
        print("="*60)
        
        comparison = pd.DataFrame(self.results).T
        comparison = comparison.sort_values('f1', ascending=False)
        
        print("\n📊 Ensemble Performance:")
        print(comparison[['precision', 'recall', 'f1', 'roc_auc']].round(4))
        
        return comparison

# Create ensemble
ensemble_exp = EnsembleExperimenter(splits, classical_exp.models)
ensemble_exp.create_voting_ensemble()
ensemble_exp.create_stacking_ensemble()
ensemble_results = ensemble_exp.train_and_evaluate()
ensemble_comparison = ensemble_exp.compare_results()

# %% [markdown]
# ### **8. Threshold Optimization**

# %%
class ThresholdOptimizer:
    """
    Optimize classification threshold for fraud detection.
    
    Finds optimal threshold based on:
    - F1 score
    - Business cost (custom cost function)
    - Precision/Recall trade-off
    """
    
    def __init__(self, model, X_val, y_val):
        """
        Initialize threshold optimizer.
        
        Args:
            model: Trained model
            X_val: Validation features
            y_val: Validation labels
        """
        self.model = model
        self.X_val = X_val
        self.y_val = y_val
        self.y_pred_proba = model.predict_proba(X_val)[:, 1]
        
    def optimize_f1(self):
        """Find threshold that maximizes F1 score"""
        precisions, recalls, thresholds = precision_recall_curve(self.y_val, self.y_pred_proba)
        
        # Calculate F1 for all thresholds
        f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-8)
        
        # Find optimal threshold
        optimal_idx = np.argmax(f1_scores)
        optimal_threshold = thresholds[optimal_idx]
        optimal_f1 = f1_scores[optimal_idx]
        
        print("\n" + "="*60)
        print("THRESHOLD OPTIMIZATION (F1 SCORE)")
        print("="*60)
        print(f"   Optimal Threshold: {optimal_threshold:.4f}")
        print(f"   Optimal F1 Score: {optimal_f1:.4f}")
        print(f"   Precision at threshold: {precisions[optimal_idx]:.4f}")
        print(f"   Recall at threshold: {recalls[optimal_idx]:.4f}")
        
        return optimal_threshold, optimal_f1
    
    def optimize_business_cost(self, fraud_cost=100, review_cost=10):
        """
        Find threshold that minimizes business cost.
        
        Args:
            fraud_cost: Cost of missing a fraud transaction
            review_cost: Cost of reviewing a false positive
        """
        thresholds = np.linspace(0, 1, 100)
        costs = []
        
        for threshold in thresholds:
            y_pred = (self.y_pred_proba >= threshold).astype(int)
            
            # Calculate costs
            false_negatives = np.sum((y_pred == 0) & (self.y_val == 1))
            false_positives = np.sum((y_pred == 1) & (self.y_val == 0))
            
            total_cost = false_negatives * fraud_cost + false_positives * review_cost
            costs.append(total_cost)
        
        optimal_idx = np.argmin(costs)
        optimal_threshold = thresholds[optimal_idx]
        optimal_cost = costs[optimal_idx]
        
        print("\n" + "="*60)
        print("THRESHOLD OPTIMIZATION (BUSINESS COST)")
        print("="*60)
        print(f"   Fraud Cost: ${fraud_cost}")
        print(f"   Review Cost: ${review_cost}")
        print(f"   Optimal Threshold: {optimal_threshold:.4f}")
        print(f"   Minimum Cost: ${optimal_cost:,.0f}")
        
        return optimal_threshold, optimal_cost
    
    def plot_threshold_analysis(self):
        """Plot threshold analysis"""
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        
        # Precision-Recall vs Threshold
        thresholds = np.linspace(0, 1, 100)
        precisions = []
        recalls = []
        f1_scores = []
        
        for threshold in thresholds:
            y_pred = (self.y_pred_proba >= threshold).astype(int)
            precisions.append(precision_score(self.y_val, y_pred, zero_division=0))
            recalls.append(recall_score(self.y_val, y_pred, zero_division=0))
            f1_scores.append(f1_score(self.y_val, y_pred, zero_division=0))
        
        axes[0, 0].plot(thresholds, precisions, label='Precision')
        axes[0, 0].plot(thresholds, recalls, label='Recall')
        axes[0, 0].plot(thresholds, f1_scores, label='F1')
        axes[0, 0].set_xlabel('Threshold')
        axes[0, 0].set_ylabel('Score')
        axes[0, 0].set_title('Precision/Recall vs Threshold')
        axes[0, 0].legend()
        axes[0, 0].grid(True, alpha=0.3)
        
        # F1 vs Threshold
        axes[0, 1].plot(thresholds, f1_scores, 'b-', linewidth=2)
        optimal_f1_threshold, optimal_f1 = self.optimize_f1()
        axes[0, 1].axvline(x=optimal_f1_threshold, color='red', linestyle='--', 
                           label=f'Optimal: {optimal_f1_threshold:.3f}')
        axes[0, 1].set_xlabel('Threshold')
        axes[0, 1].set_ylabel('F1 Score')
        axes[0, 1].set_title('F1 Score vs Threshold')
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)
        
        # Business Cost vs Threshold
        fraud_cost = 100
        review_cost = 10
        costs = []
        
        for threshold in thresholds:
            y_pred = (self.y_pred_proba >= threshold).astype(int)
            fn = np.sum((y_pred == 0) & (self.y_val == 1))
            fp = np.sum((y_pred == 1) & (self.y_val == 0))
            costs.append(fn * fraud_cost + fp * review_cost)
        
        axes[1, 0].plot(thresholds, costs, 'g-', linewidth=2)
        optimal_cost_threshold, _ = self.optimize_business_cost(fraud_cost, review_cost)
        axes[1, 0].axvline(x=optimal_cost_threshold, color='red', linestyle='--',
                           label=f'Optimal: {optimal_cost_threshold:.3f}')
        axes[1, 0].set_xlabel('Threshold')
        axes[1, 0].set_ylabel('Total Cost ($)')
        axes[1, 0].set_title('Business Cost vs Threshold')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)
        
        # Confusion Matrix at optimal threshold
        y_pred_optimal = (self.y_pred_proba >= optimal_f1_threshold).astype(int)
        cm = confusion_matrix(self.y_val, y_pred_optimal)
        
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1, 1])
        axes[1, 1].set_xlabel('Predicted')
        axes[1, 1].set_ylabel('Actual')
        axes[1, 1].set_title(f'Confusion Matrix (Threshold={optimal_f1_threshold:.3f})')
        
        plt.tight_layout()
        plt.show()

# Optimize threshold for best model
best_model_name = classical_exp.best_model_name
best_model = classical_exp.models[best_model_name]

threshold_optimizer = ThresholdOptimizer(
    best_model, 
    splits['X_val_scaled'], 
    splits['y_val']
)
threshold_optimizer.plot_threshold_analysis()
optimal_threshold, optimal_f1 = threshold_optimizer.optimize_f1()
optimal_cost_threshold, optimal_cost = threshold_optimizer.optimize_business_cost()

# %% [markdown]
# ### **9. Model Evaluation on Test Set**

# %%
class FinalEvaluator:
    """
    Final model evaluation on test set with comprehensive metrics.
    """
    
    def __init__(self, model, X_test, y_test, threshold=0.5):
        """
        Initialize final evaluator.
        
        Args:
            model: Trained model
            X_test: Test features
            y_test: Test labels
            threshold: Classification threshold
        """
        self.model = model
        self.X_test = X_test
        self.y_test = y_test
        self.threshold = threshold
        
    def evaluate(self):
        """Comprehensive model evaluation"""
        print("\n" + "="*60)
        print("FINAL MODEL EVALUATION ON TEST SET")
        print("="*60)
        
        # Predictions
        y_pred_proba = self.model.predict_proba(self.X_test)[:, 1]
        y_pred = (y_pred_proba >= self.threshold).astype(int)
        
        # Basic metrics
        accuracy = accuracy_score(self.y_test, y_pred)
        precision = precision_score(self.y_test, y_pred, zero_division=0)
        recall = recall_score(self.y_test, y_pred, zero_division=0)
        f1 = f1_score(self.y_test, y_pred, zero_division=0)
        roc_auc = roc_auc_score(self.y_test, y_pred_proba)
        pr_auc = average_precision_score(self.y_test, y_pred_proba)
        
        print(f"\n📊 Performance Metrics:")
        print(f"   Accuracy:  {accuracy:.4f}")
        print(f"   Precision: {precision:.4f}")
        print(f"   Recall:    {recall:.4f}")
        print(f"   F1 Score:  {f1:.4f}")
        print(f"   ROC AUC:   {roc_auc:.4f}")
        print(f"   PR AUC:    {pr_auc:.4f}")
        
        # Confusion Matrix
        cm = confusion_matrix(self.y_test, y_pred)
        print(f"\n📊 Confusion Matrix:")
        print(f"   True Negatives:  {cm[0, 0]:,}")
        print(f"   False Positives: {cm[0, 1]:,}")
        print(f"   False Negatives: {cm[1, 0]:,}")
        print(f"   True Positives:  {cm[1, 1]:,}")
        
        # Classification Report
        print(f"\n📊 Classification Report:")
        print(classification_report(self.y_test, y_pred, 
                                    target_names=['Non-Fraud', 'Fraud']))
        
        # Business metrics
        fraud_cost = 100
        review_cost = 10
        total_cost = cm[1, 0] * fraud_cost + cm[0, 1] * review_cost
        savings = cm[1, 1] * fraud_cost - cm[0, 1] * review_cost
        
        print(f"\n💰 Business Impact:")
        print(f"   Fraud caught: {cm[1, 1]:,} (saved ${cm[1, 1] * fraud_cost:,})")
        print(f"   False positives: {cm[0, 1]:,} (cost ${cm[0, 1] * review_cost:,})")
        print(f"   Missed fraud: {cm[1, 0]:,} (cost ${cm[1, 0] * fraud_cost:,})")
        print(f"   Net savings: ${savings:,}")
        print(f"   Total cost: ${total_cost:,}")
        
        # Store results
        self.results = {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'roc_auc': roc_auc,
            'pr_auc': pr_auc,
            'confusion_matrix': cm,
            'business_metrics': {
                'fraud_caught': cm[1, 1],
                'false_positives': cm[0, 1],
                'missed_fraud': cm[1, 0],
                'savings': savings,
                'total_cost': total_cost
            }
        }
        
        return self.results
    
    def plot_results(self):
        """Visualize evaluation results"""
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        
        y_pred_proba = self.model.predict_proba(self.X_test)[:, 1]
        y_pred = (y_pred_proba >= self.threshold).astype(int)
        
        # 1. Confusion Matrix
        cm = self.results['confusion_matrix']
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, 0],
                   xticklabels=['Non-Fraud', 'Fraud'],
                   yticklabels=['Non-Fraud', 'Fraud'])
        axes[0, 0].set_title('Confusion Matrix')
        axes[0, 0].set_xlabel('Predicted')
        axes[0, 0].set_ylabel('Actual')
        
        # 2. ROC Curve
        fpr, tpr, _ = roc_curve(self.y_test, y_pred_proba)
        roc_auc = self.results['roc_auc']
        
        axes[0, 1].plot(fpr, tpr, 'b-', label=f'ROC (AUC = {roc_auc:.4f})')
        axes[0, 1].plot([0, 1], [0, 1], 'r--', alpha=0.5)
        axes[0, 1].set_xlabel('False Positive Rate')
        axes[0, 1].set_ylabel('True Positive Rate')
        axes[0, 1].set_title('ROC Curve')
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)
        
        # 3. Precision-Recall Curve
        precision, recall, _ = precision_recall_curve(self.y_test, y_pred_proba)
        pr_auc = self.results['pr_auc']
        
        axes[0, 2].plot(recall, precision, 'g-', label=f'PR (AUC = {pr_auc:.4f})')
        axes[0, 2].axhline(y=self.results['precision'], color='red', linestyle='--', 
                           label=f'Precision: {self.results["precision"]:.4f}')
        axes[0, 2].axvline(x=self.results['recall'], color='blue', linestyle='--',
                           label=f'Recall: {self.results["recall"]:.4f}')
        axes[0, 2].set_xlabel('Recall')
        axes[0, 2].set_ylabel('Precision')
        axes[0, 2].set_title('Precision-Recall Curve')
        axes[0, 2].legend()
        axes[0, 2].grid(True, alpha=0.3)
        
        # 4. Probability Distribution
        fraud_probs = y_pred_proba[self.y_test == 1]
        non_fraud_probs = y_pred_proba[self.y_test == 0]
        
        axes[1, 0].hist(non_fraud_probs, bins=50, alpha=0.7, label='Non-Fraud', density=True)
        axes[1, 0].hist(fraud_probs, bins=50, alpha=0.7, label='Fraud', density=True)
        axes[1, 0].axvline(x=self.threshold, color='red', linestyle='--', 
                           label=f'Threshold: {self.threshold:.3f}')
        axes[1, 0].set_xlabel('Predicted Probability')
        axes[1, 0].set_ylabel('Density')
        axes[1, 0].set_title('Probability Distribution by Class')
        axes[1, 0].legend()
        
        # 5. Performance Metrics Bar Chart
        metrics = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC AUC', 'PR AUC']
        values = [self.results[m.lower().replace(' ', '_')] for m in metrics]
        
        axes[1, 1].bar(metrics, values, color='skyblue')
        axes[1, 1].set_ylim([0, 1])
        axes[1, 1].set_ylabel('Score')
        axes[1, 1].set_title('Performance Metrics')
        axes[1, 1].tick_params(axis='x', rotation=45)
        
        # 6. Business Impact
        business = self.results['business_metrics']
        categories = ['Fraud Caught', 'False Positives', 'Missed Fraud']
        values = [business['fraud_caught'], business['false_positives'], business['missed_fraud']]
        colors = ['green', 'orange', 'red']
        
        axes[1, 2].bar(categories, values, color=colors)
        axes[1, 2].set_ylabel('Count')
        axes[1, 2].set_title('Business Impact')
        axes[1, 2].tick_params(axis='x', rotation=45)
        
        # Add savings text
        savings_text = f"Net Savings: ${business['savings']:,}"
        axes[1, 2].text(0.5, 0.95, savings_text, transform=axes[1, 2].transAxes,
                       ha='center', va='top', fontweight='bold',
                       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        plt.tight_layout()
        plt.show()

# Evaluate best model
final_evaluator = FinalEvaluator(
    best_model, 
    splits['X_test_scaled'], 
    splits['y_test'], 
    threshold=optimal_threshold
)
final_results = final_evaluator.evaluate()
final_evaluator.plot_results()

# %% [markdown]
# ### **10. Model Interpretability**

# %%
class ModelExplainer:
    """
    Explain model predictions using SHAP and LIME.
    """
    
    def __init__(self, model, X_train, feature_names):
        """
        Initialize model explainer.
        
        Args:
            model: Trained model
            X_train: Training features (for background distribution)
            feature_names: List of feature names
        """
        self.model = model
        self.X_train = X_train
        self.feature_names = feature_names
        self.shap_values = None
        
    def explain_with_shap(self, X_explain, n_samples=100):
        """
        Explain predictions using SHAP.
        
        Args:
            X_explain: Samples to explain
            n_samples: Number of background samples
        """
        print("\n" + "="*60)
        print("SHAP EXPLANATION")
        print("="*60)
        
        # Create SHAP explainer
        if hasattr(self.model, 'feature_importances_'):
            # Tree-based models
            explainer = shap.TreeExplainer(self.model)
            self.shap_values = explainer.shap_values(X_explain)
        else:
            # Other models (use KernelExplainer with background)
            background = shap.sample(self.X_train, n_samples)
            explainer = shap.KernelExplainer(self.model.predict_proba, background)
            self.shap_values = explainer.shap_values(X_explain)
        
        # Summary plot
        print("\n📊 SHAP Summary Plot:")
        shap.summary_plot(self.shap_values, X_explain, feature_names=self.feature_names, show=False)
        plt.tight_layout()
        plt.show()
        
        # Feature importance
        shap_importance = np.abs(self.shap_values).mean(axis=0)
        importance_df = pd.DataFrame({
            'feature': self.feature_names,
            'importance': shap_importance
        }).sort_values('importance', ascending=False)
        
        print("\n📊 Top 10 Features by SHAP Importance:")
        for i, row in importance_df.head(10).iterrows():
            print(f"   {i+1}. {row['feature']}: {row['importance']:.4f}")
        
        return importance_df
    
    def explain_with_lime(self, X_instance):
        """
        Explain a single prediction using LIME.
        
        Args:
            X_instance: Single instance to explain
        """
        print("\n" + "="*60)
        print("LIME EXPLANATION FOR SINGLE INSTANCE")
        print("="*60)
        
        # Create LIME explainer
        explainer = lime.lime_tabular.LimeTabularExplainer(
            self.X_train,
            feature_names=self.feature_names,
            class_names=['Non-Fraud', 'Fraud'],
            mode='classification',
            random_state=42
        )
        
        # Explain instance
        exp = explainer.explain_instance(
            X_instance.flatten(),
            self.model.predict_proba,
            num_features=10,
            top_labels=1
        )
        
        # Show explanation
        exp.show_in_notebook(show_table=True)
        exp.as_pyplot_figure()
        plt.tight_layout()
        plt.show()
        
        return exp
    
    def plot_feature_interaction(self, feature1, feature2):
        """
        Plot SHAP interaction between two features.
        """
        if self.shap_values is None:
            print("Please run SHAP explanation first")
            return
        
        # Get indices for features
        idx1 = self.feature_names.index(feature1)
        idx2 = self.feature_names.index(feature2)
        
        # Extract SHAP values for these features
        shap_interaction = self.shap_values[:, idx1] * self.shap_values[:, idx2]
        
        plt.figure(figsize=(10, 6))
        plt.scatter(self.X_train[:, idx1], self.X_train[:, idx2], 
                   c=shap_interaction, cmap='RdBu', alpha=0.6)
        plt.colorbar(label='SHAP Interaction')
        plt.xlabel(feature1)
        plt.ylabel(feature2)
        plt.title(f'SHAP Interaction: {feature1} vs {feature2}')
        plt.tight_layout()
        plt.show()

# Explain best model
explainer = ModelExplainer(
    best_model,
    splits['X_train_scaled'][:1000],  # Sample for background
    preparator.feature_names
)

# SHAP explanation (may take time for large datasets)
X_sample = splits['X_test_scaled'][:100]  # Sample for explanation
importance_df = explainer.explain_with_shap(X_sample)

# LIME explanation for a single instance
X_single = splits['X_test_scaled'][0:1]
exp = explainer.explain_with_lime(X_single)

# %% [markdown]
# ### **11. Model Saving and Export**

# %%
class ModelSaver:
    """
    Save trained model and all artifacts for deployment.
    """
    
    def __init__(self, model, preprocessor, threshold, metadata):
        """
        Initialize model saver.
        
        Args:
            model: Trained model
            preprocessor: Data preprocessor (scaler, etc.)
            threshold: Optimal threshold
            metadata: Model metadata
        """
        self.model = model
        self.preprocessor = preprocessor
        self.threshold = threshold
        self.metadata = metadata
        
    def save_model(self, path='../artifacts/models/'):
        """
        Save model and artifacts.
        
        Args:
            path: Directory to save artifacts
        """
        print("\n" + "="*60)
        print("SAVING MODEL AND ARTIFACTS")
        print("="*60)
        
        # Create directory if it doesn't exist
        os.makedirs(path, exist_ok=True)
        
        # Save model
        model_path = os.path.join(path, 'fraud_detection_model.pkl')
        with open(model_path, 'wb') as f:
            pickle.dump(self.model, f)
        print(f"✅ Model saved to: {model_path}")
        
        # Save preprocessor
        preprocessor_path = os.path.join(path, 'preprocessor.pkl')
        with open(preprocessor_path, 'wb') as f:
            pickle.dump(self.preprocessor, f)
        print(f"✅ Preprocessor saved to: {preprocessor_path}")
        
        # Save threshold
        threshold_path = os.path.join(path, 'threshold.json')
        with open(threshold_path, 'w') as f:
            json.dump({'threshold': self.threshold}, f)
        print(f"✅ Threshold saved to: {threshold_path}")
        
        # Save metadata
        metadata_path = os.path.join(path, 'metadata.json')
        with open(metadata_path, 'w') as f:
            json.dump(self.metadata, f, indent=2)
        print(f"✅ Metadata saved to: {metadata_path}")
        
        # Save feature names
        feature_path = os.path.join(path, 'feature_names.json')
        with open(feature_path, 'w') as f:
            json.dump(self.metadata['feature_names'], f)
        print(f"✅ Feature names saved to: {feature_path}")
        
        print(f"\n📦 All artifacts saved to: {path}")

# Prepare metadata
metadata = {
    'model_name': best_model_name,
    'feature_names': preparator.feature_names,
    'optimal_threshold': optimal_threshold,
    'test_performance': {
        'f1': final_results['f1'],
        'precision': final_results['precision'],
        'recall': final_results['recall'],
        'roc_auc': final_results['roc_auc'],
        'pr_auc': final_results['pr_auc']
    },
    'business_metrics': final_results['business_metrics'],
    'training_date': datetime.now().isoformat(),
    'version': '1.0.0'
}

# Save model
model_saver = ModelSaver(
    best_model,
    preparator.scaler,
    optimal_threshold,
    metadata
)
model_saver.save_model()

# %% [markdown]
# ### **12. Summary and Next Steps**

# %%
print("\n" + "="*60)
print("MODEL EXPERIMENTATION COMPLETE - SUMMARY")
print("="*60)

print(f"""
🏆 Best Model: {best_model_name}
   F1 Score: {final_results['f1']:.4f}
   Precision: {final_results['precision']:.4f}
   Recall: {final_results['recall']:.4f}
   ROC AUC: {final_results['roc_auc']:.4f}

💰 Business Impact:
   • Fraud caught: {final_results['business_metrics']['fraud_caught']:,}
   • False positives: {final_results['business_metrics']['false_positives']:,}
   • Missed fraud: {final_results['business_metrics']['missed_fraud']:,}
   • Net savings: ${final_results['business_metrics']['savings']:,}

📊 Models Evaluated:
   • Classical ML: {len(classical_exp.models)} models
   • Anomaly Detection: {len(anomaly_exp.models)} models
   • Deep Learning: {len(dl_exp.models)} models
   • Ensemble: {len(ensemble_exp.ensemble_models)} models

🎯 Optimal Threshold: {optimal_threshold:.4f} (maximizes F1)

📦 Artifacts Saved:
   • Model: ../artifacts/models/fraud_detection_model.pkl
   • Preprocessor: ../artifacts/models/preprocessor.pkl
   • Metadata: ../artifacts/models/metadata.json

🚀 Next Steps:
   1. Proceed to Notebook 05: Model Evaluation (detailed analysis)
   2. Validate model on hold-out test sets
   3. Prepare for production deployment
   4. Set up monitoring and drift detection
""")